<a href="https://colab.research.google.com/github/ShivaniYadav354/CCTV-anomaly-detection/blob/main/CCTV_Anomaly_Detection_and_surveillance_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Connecting Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Unzip UCF Dataset


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/archive.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
import os
print(os.listdir('/content/dataset'))

['Test', 'Train']


In [ ]:
import os

base_path = "/content/dataset/Train"

classes = ['Arson', 'Fighting', 'NormalVideos', 'Shoplifting', 'Vandalism']

for cls in classes:
    print(cls, ":", len(os.listdir(os.path.join(base_path, cls))))

Arson : 24421
Fighting : 24684
NormalVideos : 947768
Shoplifting : 24835
Vandalism : 13626


Filtering Dataset

In [ ]:
import cv2

def is_blurry(image_path, threshold=100):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    variance = cv2.Laplacian(gray, cv2.CV_64F).var()

    return variance < threshold  # True = blurry

In [ ]:
def is_bad_brightness(image_path, low=50, high=200):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    mean = gray.mean()

    return mean < low or mean > high

After Filtering Dataset

In [ ]:
import os

base_path = "/content/dataset/Train"

filtered_path = "/content/dataset/Filtered_Train"
os.makedirs(filtered_path, exist_ok=True)

classes = ['Arson', 'Fighting', 'Normal', 'Shoplifting', 'Vandalism']

for cls in classes:
    src_folder = os.path.join(base_path, cls)
    dst_folder = os.path.join(filtered_path, cls)
    os.makedirs(dst_folder, exist_ok=True)

    images = os.listdir(src_folder)

    count = 0

    for img_name in images:
        img_path = os.path.join(src_folder, img_name)

        try:
            if not is_blurry(img_path) and not is_bad_brightness(img_path):
                img = cv2.imread(img_path)
                cv2.imwrite(os.path.join(dst_folder, img_name), img)
                count += 1
        except:
            continue

    print(f"{cls} → kept {count} images")

Arson → kept 22781 images
Fighting → kept 23902 images
Normal → kept 938328 images
Shoplifting → kept 24793 images
Vandalism → kept 11655 images


Zip Filtering Dataset

In [ ]:
import shutil

shutil.make_archive(
    "/content/filtered_dataset",
    'zip',
    "/content/filtered_dataset"
)

'/content/filtered_dataset.zip'

Saving Dataset in Drive


In [ ]:
import shutil

shutil.move(
    "/content/filtered_dataset.zip",
    "/content/drive/MyDrive/filtered_dataset.zip"
)

print("✅ Dataset saved permanently in Drive")

✅ Dataset saved permanently in Drive


Balancing Dataset


In [ ]:
import os, random, shutil

BASE_PATH = "/content/filtered_dataset"
FINAL_PATH = "/content/final_dataset"

CLASS_LIMITS = {
    "Normal": 1000,
    "Arson": 250,
    "Fighting": 250,
    "Shoplifting": 250,
    "Vandalism": 250
}

# Clear old dataset (IMPORTANT)
if os.path.exists(FINAL_PATH):
    shutil.rmtree(FINAL_PATH)

os.makedirs(FINAL_PATH)

for cls, limit in CLASS_LIMITS.items():
    src = os.path.join(BASE_PATH, cls)
    dst = os.path.join(FINAL_PATH, cls)

    os.makedirs(dst)

    images = os.listdir(src)

    # IMPORTANT: random selection every run
    selected = random.sample(images, min(limit, len(images)))

    for img in selected:
        shutil.copy(os.path.join(src, img), os.path.join(dst, img))

    print(f"{cls} → {len(selected)} images selected")

FileNotFoundError: [Errno 2] No such file or directory: '/content/filtered_dataset/Normal'

Balancing UCF Dataset

In [ ]:
import os, random, shutil

BASE_PATH = "/content/dataset/UCF_Crime_Dataset"
FINAL_PATH = "/content/raw_final_dataset"

CLASS_LIMITS = {
    "Normal": 1000,
    "Arson": 250,
    "Fighting": 250,
    "Shoplifting": 250,
    "Vandalism": 250
}

if os.path.exists(FINAL_PATH):
    shutil.rmtree(FINAL_PATH)

os.makedirs(FINAL_PATH)

for cls, limit in CLASS_LIMITS.items():
    src = os.path.join(BASE_PATH, cls)
    dst = os.path.join(FINAL_PATH, cls)

    os.makedirs(dst)

    images = os.listdir(src)

    selected = random.sample(images, min(limit, len(images)))

    for img in selected:
        shutil.copy(os.path.join(src, img), os.path.join(dst, img))

    print(f"{cls} → {len(selected)} images selected")

Spliting

In [ ]:
from sklearn.model_selection import train_test_split
import os, shutil

BASE_PATH = "/content/final_dataset"
SPLIT_PATH = "/content/final_split"

# Clear old split
if os.path.exists(SPLIT_PATH):
    shutil.rmtree(SPLIT_PATH)

for cls in os.listdir(BASE_PATH):
    src = os.path.join(BASE_PATH, cls)
    images = os.listdir(src)

    # Split: 70% train, 15% val, 15% test
    train, temp = train_test_split(images, test_size=0.3, random_state=42)
    val, test = train_test_split(temp, test_size=0.5, random_state=42)

    splits = {
        "train": train,
        "val": val,
        "test": test
    }

    for split, img_list in splits.items():
        dst = os.path.join(SPLIT_PATH, split, cls)
        os.makedirs(dst, exist_ok=True)

        for img in img_list:
            shutil.copy(os.path.join(src, img), os.path.join(dst, img))

    print(f" {cls} split done")

Spliting the UCF Raw Dataset

In [ ]:
import os, random, shutil
from sklearn.model_selection import train_test_split

BASE_PATH = "/content/raw_final_dataset"
SPLIT_PATH = "/content/raw_split"

if os.path.exists(SPLIT_PATH):
    shutil.rmtree(SPLIT_PATH)

for cls in os.listdir(BASE_PATH):
    images = os.listdir(os.path.join(BASE_PATH, cls))

    train_imgs, temp_imgs = train_test_split(images, test_size=0.3, random_state=42)
    val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.5, random_state=42)

    for split, img_list in zip(["train", "val", "test"], [train_imgs, val_imgs, test_imgs]):
        dst = os.path.join(SPLIT_PATH, split, cls)
        os.makedirs(dst, exist_ok=True)

        for img in img_list:
            src = os.path.join(BASE_PATH, cls, img)
            shutil.copy(src, dst)

    print(f"✅ {cls} split done")

In [ ]:
for split in ["train","val","test"]:
    print(f"\n--- {split.upper()} ---")

    for cls in os.listdir(f"/content/final_split/{split}"):
        count = len(os.listdir(f"/content/final_split/{split}/{cls}"))
        print(cls, "→", count)

Feature extract

In [ ]:
import numpy as np
import cv2
import os
import random

SEQ_LENGTH = 20
IMG_SIZE = (224, 224)

def augment_image(img):
    if random.random() > 0.5:
        img = cv2.flip(img, 1)

    if random.random() > 0.5:
        img = cv2.convertScaleAbs(img, alpha=1.2, beta=20)

    if random.random() > 0.5:
        img = cv2.GaussianBlur(img, (3,3), 0)

    return img


def load_sequences(base_path):
    X, y = [], []

    class_names = ["Arson", "Fighting", "NormalVideos", "Shoplifting", "Vandalism"]

    for label, cls in enumerate(class_names):
        class_path = os.path.join(base_path, cls)

        if not os.path.exists(class_path):
            continue

        images = os.listdir(class_path)
        random.shuffle(images)

        for i in range(0, len(images) - SEQ_LENGTH, SEQ_LENGTH):
            seq = []

            for j in range(SEQ_LENGTH):
                img_path = os.path.join(class_path, images[i + j])
                img = cv2.imread(img_path)

                if img is None:
                    continue

                img = cv2.resize(img, IMG_SIZE)


                img = augment_image(img)

                img = img / 255.0
                seq.append(img)

            if len(seq) == SEQ_LENGTH:
                X.append(seq)
                y.append(label)

    return np.array(X), np.array(y)

In [ ]:
from tensorflow.keras.utils import to_categorical

# FILTERED
X_train_f, y_train_f = load_sequences("/content/final_split/train")
X_val_f, y_val_f = load_sequences("/content/final_split/val")
X_test_f, y_test_f = load_sequences("/content/final_split/test")

y_train_f = to_categorical(y_train_f, 5)
y_val_f = to_categorical(y_val_f, 5)
y_test_f = to_categorical(y_test_f, 5)

# RAW
X_train_r, y_train_r = load_sequences("/content/raw_split/train")
X_val_r, y_val_r = load_sequences("/content/raw_split/val")
X_test_r, y_test_r = load_sequences("/content/raw_split/test")

y_train_r = to_categorical(y_train_r, 5)
y_val_r = to_categorical(y_val_r, 5)
y_test_r = to_categorical(y_test_r, 5)

In [ ]:
def build_model():
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import TimeDistributed, LSTM, Dense, GlobalAveragePooling2D, Dropout
    from tensorflow.keras.applications import MobileNetV2

    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(128,128,3)
    )

    # Freeze base
    base_model.trainable = False

    # 🔥 Fine-tune last layers (INSIDE function)
    for layer in base_model.layers[-20:]:
        layer.trainable = True

    model = Sequential([
        TimeDistributed(base_model, input_shape=(5,128,128,3)),
        TimeDistributed(GlobalAveragePooling2D()),
        LSTM(64),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dense(5, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

Sequence Creation of UCF Raw Dataset

In [ ]:
model_filtered = build_model()

model_filtered.fit(X_train_f, y_train_f,
                   validation_data=(X_val_f, y_val_f),
                   epochs=10)

loss_f, acc_filtered = model_filtered.evaluate(X_test_f, y_test_f)

In [ ]:
model_raw = build_model()

model_raw.fit(X_train_r, y_train_r,
              validation_data=(X_val_r, y_val_r),
              epochs=10)

loss_r, acc_raw = model_raw.evaluate(X_test_r, y_test_r)

In [ ]:
model_filtered.summary()
model_raw.summary()

In [ ]:
history_filtered = model_filtered.fit(
    X_train_f, y_train_f,
    validation_data=(X_val_f, y_val_f),
    epochs=10,
    batch_size=8
)
history_raw = model_raw.fit(
    X_train_r, y_train_r,
    validation_data=(X_val_r, y_val_r),
    epochs=10,
    batch_size=8
)


UCF Raw Dataset Accuracy

In [ ]:
loss_r, acc_raw = model_raw.evaluate(X_test_r, y_test_r)

print("Raw Dataset Accuracy:", acc_raw)

In [ ]:
loss_f, acc_filtered = model_filtered.evaluate(X_test_f, y_test_f)

print("Filtered Dataset Accuracy:", acc_filtered)

In [ ]:
print("\n📊 FINAL COMPARISON")
print(f"Filtered Accuracy: {acc_filtered}")
print(f"Raw Accuracy     : {acc_raw}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history_filtered.history['accuracy'], label='Train Accuracy')
plt.plot(history_filtered.history['val_accuracy'], label='Val Accuracy')

plt.title('Filtered Dataset Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()



plt.plot(history_raw.history['accuracy'], label='Train Accuracy')
plt.plot(history_raw.history['val_accuracy'], label='Val Accuracy')

plt.title('Raw Dataset Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
plt.plot(history_filtered.history['val_accuracy'], label='Filtered')
plt.plot(history_raw.history['val_accuracy'], label='Raw')

plt.title('Filtered vs Raw Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
def predict_sequence(seq, model):
    seq = np.expand_dims(seq, axis=0)
    pred = model.predict(seq)
    return np.argmax(pred)

In [ ]:
class_names = ["Arson", "Fighting", "Normal", "Shoplifting", "Vandalism"]

# Take user inputs
predicted_input = input("Enter predicted class: ")
target_class = input("Enter target anomaly class: ")

# Clean inputs
predicted_input = predicted_input.strip()
target_class = target_class.strip()

# Validation
if predicted_input not in class_names:
    print("❌ Invalid predicted class")
elif target_class not in class_names:
    print("❌ Invalid target class")
else:
    print("Predicted:", predicted_input)
    print("Target:", target_class)

    if predicted_input == target_class:
        print("Result: Anomaly Detected")
    else:
        print("Result: No Anomaly")

In [ ]:
def predict_from_images(image_paths, model):
    seq = []

    for path in image_paths:
        img = cv2.imread(path)
        img = cv2.resize(img, (128,128))
        img = img / 255.0
        seq.append(img)

    seq = np.array(seq)
    seq = np.expand_dims(seq, axis=0)

    pred = model.predict(seq)
    return np.argmax(pred)

In [ ]:
class_names = ["Arson", "Fighting", "Normal", "Shoplifting", "Vandalism"]

folder = "/content/final_split/test/Shoplifting"
imgs = sorted(os.listdir(folder))[:5]

image_paths = [os.path.join(folder, img) for img in imgs]
pred = predict_from_images(image_paths, model_filtered)
print("🚨 Detected Activity:", class_names[pred])

In [ ]:
import shutil
import os

# Path to your balanced dataset folder
balanced_dataset_path = "/content/final_dataset"   # or "/content/raw_final_dataset"

# Where to save the zip
zip_output_path = "/content/drive/MyDrive/ CCTV_Project/balanced_dataset.zip"

# Create zip
shutil.make_archive(zip_output_path.replace('.zip', ''), 'zip', balanced_dataset_path)

print(f"✅ Balanced dataset saved as: {zip_output_path}")

In [ ]:
# After training
model_filtered.save_weights("/content/drive/MyDrive/ CCTV_Project/filtered_model.weights.h5")
model_raw.save_weights("/content/drive/MyDrive/ CCTV_Project/raw_model.weights.h5")
print("✅ Weights saved as .h5 files")

In [ ]:
model_filtered.save("/content/drive/MyDrive/ CCTV_Project/filtered_model_full.h5")
model_raw.save("/content/drive/MyDrive/ CCTV_Project/raw_model_full.h5")

In [ ]:
import pickle
import numpy as np

# Extract weights as list of numpy arrays
weights = model_filtered.get_weights()
with open("/content/drive/MyDrive/ CCTV_Project/filtered_weights.pkl", "wb") as f:
    pickle.dump(weights, f)

# To load later:
# model_filtered.set_weights(pickle.load(open("filtered_weights.pkl", "rb")))